# Non-Sequitur Prompt Engineering Notebook

This notebook is for iterating on the non-sequitur insert-only prompt before running a larger batch.

It uses the same live generation path as the batch runner:

- `non-sequiturs.json`
- `prompts/non_sequitur_insert.md`
- `scripts/build_non_sequitur_dataset.py`
- `src/trace_augmentor.py`

Unlike the earlier scaffold, this notebook:

1. samples from the **full merged dataset** used in the OLMo SFT workflow,
2. filters to rows with **full-row tokens < 14000**,
3. renders the exact prompt for one sampled insertion,
4. calls OpenRouter live for structured output,
5. validates the returned blocks,
6. computes mask targets,
7. and can batch-run a small real sample for prompt review.

In [26]:
import os
from pathlib import Path


def _load_env_file(path: str | Path) -> dict[str, str]:
    values: dict[str, str] = {}
    env_path = Path(path)
    if not env_path.exists():
        return values
    for raw_line in env_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


NOTEBOOK_PATH = Path.cwd().resolve() / "non_sequitur_prompt_engineering.ipynb"
if not NOTEBOOK_PATH.exists():
    NOTEBOOK_PATH = (
        Path.cwd().resolve()
        / "augmentation"
        / "steer_exec_augmentation_bundle"
        / "notebooks"
        / "non_sequitur_prompt_engineering.ipynb"
    )
assert NOTEBOOK_PATH.exists(), "Could not locate the non-sequitur notebook path."

NOTEBOOK_DIR = NOTEBOOK_PATH.parent
BUNDLE_ROOT = NOTEBOOK_DIR.parent
REPO_ROOT = BUNDLE_ROOT.parent.parent
ENV_FILE = REPO_ROOT / ".env"
ENV_VALUES = _load_env_file(ENV_FILE)

OPENROUTER_API_KEY = (
    os.environ.get("OPENROUTER_API_KEY")
    or ENV_VALUES.get("OPENROUTER_API_KEY")
    or os.environ.get("OPEN_ROUTER_KEY")
    or ENV_VALUES.get("OPEN_ROUTER_KEY")
)
assert OPENROUTER_API_KEY, f"No OpenRouter key found in env vars or {ENV_FILE}"
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

In [27]:
from __future__ import annotations

import asyncio
import importlib
import json
import random
import sys
from typing import Any

import tiktoken

if str(BUNDLE_ROOT) not in sys.path:
    sys.path.insert(0, str(BUNDLE_ROOT))

import src.non_sequitur_builder as non_sequitur_builder
import src.non_sequitur_source as non_sequitur_source
import src.trace_augmentor as trace_augmentor
import scripts.build_non_sequitur_dataset as non_sequitur_runner

non_sequitur_builder = importlib.reload(non_sequitur_builder)
non_sequitur_source = importlib.reload(non_sequitur_source)
trace_augmentor = importlib.reload(trace_augmentor)
non_sequitur_runner = importlib.reload(non_sequitur_runner)

## Configuration

The source path below is the full merged dataset used in the SFT pipeline, not the earlier 523-row subset.

In [ ]:
SOURCE_PATH = Path(
    os.environ.get(
        "NON_SEQUITUR_INPUT_PATH",
        str(
            REPO_ROOT
            / "output_transform_async_16384"
            / "transformed_subset_analysis"
            / "merged_with_output_transformed_output.jsonl"
        ),
    )
)
INTERVENTIONS_PATH = BUNDLE_ROOT / "non-sequiturs.json"
PROMPTS_DIR = BUNDLE_ROOT / "prompts"
OPENROUTER_MODEL = os.environ.get("NON_SEQUITUR_MODEL", "openai/gpt-oss-20b")
OPENROUTER_MAX_TOKENS = int(os.environ.get("NON_SEQUITUR_MAX_TOKENS", "2000"))
OPENROUTER_TEMPERATURE = float(os.environ.get("NON_SEQUITUR_TEMPERATURE", "0.4"))
SEED = int(os.environ.get("NON_SEQUITUR_SEED", "127"))
MAX_SOURCE_TOKENS = 14000
EXEC_TOKEN_LIMIT = 512
STYLE_WINDOW_PAIRS = 2
BATCH_SAMPLE_SIZE = 15
TOKENIZER_NAME = None

ENCODING = tiktoken.get_encoding("cl100k_base")
TOKEN_COUNTER = trace_augmentor.TokenCounter(TOKENIZER_NAME)
rng = random.Random(SEED)

assert SOURCE_PATH.exists(), f"Input dataset not found: {SOURCE_PATH}"

In [51]:
rows = trace_augmentor.load_jsonl(SOURCE_PATH)
interventions_obj = trace_augmentor.load_json(INTERVENTIONS_PATH)
non_sequitur_runner.validate_interventions_obj(interventions_obj=interventions_obj)
prepared_rows, prep_summary = non_sequitur_source.prepare_source_rows(
    rows=rows,
    encoding=ENCODING,
    max_source_tokens=MAX_SOURCE_TOKENS,
)
print(f"source path: {SOURCE_PATH}")
print(f"raw rows: {len(rows)}")
print(json.dumps(prep_summary.to_json(), indent=2))
print(f"loaded interventions: {len(interventions_obj['interventions'])}")

source path: /users/PAA0201/ollieproudman/work/DecomposedReasoning/BuildSFTDataset/output_transform_async_16384/transformed_subset_analysis/merged_with_output_transformed_output.jsonl
raw rows: 2193
{
  "total_rows": 2193,
  "eligible_rows": 1975,
  "skipped_over_token_limit": 15,
  "skipped_invalid_trace": 0,
  "skipped_short_trace": 0,
  "skipped_invalid_blocks": 203
}
loaded interventions: 25


## Inspect eligible rows

In [52]:
preview_rows = rng.sample(prepared_rows, k=min(15, len(prepared_rows)))
for prepared_row in preview_rows:
    print("=" * 100)
    print(
        {
            "row_index": prepared_row.row_index,
            "row_id": prepared_row.row["id"],
            "dataset_source": prepared_row.row.get("dataset_source", ""),
            "full_row_tokens": prepared_row.complete_prompt_token_count,
            "pair_count": len(prepared_row.parsed_blocks) // 2,
        }
    )
    user_messages = [
        message
        for message in prepared_row.row["messages"]
        if message.get("role") == "user" and isinstance(message.get("content"), str)
    ]
    if user_messages:
        print(user_messages[-1]["content"][:300])

{'row_index': 103, 'row_id': 'correct-python-sft-187k-x16-thoughts-filtered-decontam-v2_ca3a98b8-7d39-4c53-b33a-dd05b5d26c2b', 'dataset_source': 'saumyamalik/correct-python-sft-187k-x16-thoughts-filtered-decontam-v2', 'full_row_tokens': 1323, 'pair_count': 17}
You are tasked with implementing a function named `acquire_resources` that simulates the acquisition of resources for a production system. The function takes a list of entry limit attributes as input. It should log messages indicating whether each resource acquisition complies with the entry limit a
{'row_index': 2078, 'row_id': 'correct-python-sft-187k-x16-thoughts-filtered-decontam-v2_14cd4022-506e-4767-a46f-7fa5bac57a72', 'dataset_source': 'saumyamalik/correct-python-sft-187k-x16-thoughts-filtered-decontam-v2', 'full_row_tokens': 1371, 'pair_count': 16}
You are tasked with creating a function that generates a string representation of a 'Standin' file with specified attributes. The function should take a variable number of keyw

## Pick one row and render the exact prompt

In [53]:
prepared_row = prepared_rows[rng.randrange(len(prepared_rows))]
intervention_spec = dict(trace_augmentor.choose_intervention(interventions_obj, rng))
plan = non_sequitur_builder.build_non_sequitur_plan(
    record_index=prepared_row.row_index,
    record=prepared_row.row,
    parsed_blocks=prepared_row.parsed_blocks,
    intervention_spec=intervention_spec,
    seed=SEED * 10_000 + prepared_row.row_index,
)
prefix_blocks = trace_augmentor.slice_pairs(
    prepared_row.parsed_blocks,
    plan.cut_after_pairs,
)
prompt_values = trace_augmentor.build_prompt_values(
    record=dict(prepared_row.row),
    all_blocks=prepared_row.parsed_blocks,
    prefix_blocks=prefix_blocks,
    intervention_spec=dict(intervention_spec),
    intervention_variant=plan.variant,
    pairs_to_generate_k=plan.pairs_generated,
    exec_token_limit=EXEC_TOKEN_LIMIT,
    style_window_pairs=STYLE_WINDOW_PAIRS,
    validation_feedback="",
)
prompt_path = trace_augmentor.choose_prompt_path(
    prompts_dir=PROMPTS_DIR,
    mode="insert",
    intervention_spec=intervention_spec,
)
system_prompt = (PROMPTS_DIR / "system.md").read_text(encoding="utf-8")
user_prompt = trace_augmentor.render_prompt(path=prompt_path, values=prompt_values)
print(
    {
        "row_index": prepared_row.row_index,
        "row_id": prepared_row.row["id"],
        "theme": intervention_spec["name"],
        "cut_after_pairs": plan.cut_after_pairs,
        "pairs_generated": plan.pairs_generated,
        "variant": plan.variant,
    }
)
print("\nSYSTEM PROMPT:\n")
print(system_prompt[:1600], "..." if len(system_prompt) > 1600 else "")
print("\n" + "=" * 100 + "\n")
print("USER PROMPT:\n")
print(user_prompt[:5000], "..." if len(user_prompt) > 5000 else "")

{'row_index': 1569, 'row_id': 'OpenThoughts3-full-filtered-science-decontam-v2_33918', 'theme': 'Invent a temporary rule for the current variable', 'cut_after_pairs': 3, 'pairs_generated': 3, 'variant': 'Declare a short local rule around the current symbol'}

SYSTEM PROMPT:

# System Prompt

You are editing an existing reasoning trace that is written as alternating **steer** and **exec** blocks.

Your job is to generate **only a short intervention window** that could plausibly have been produced by the **same internal reasoner** at the given point in the trace.

## Behavioral goals

- Stay specific to the problem.
- Match the local tone, detail level, and pacing of the prefix.
- Make the intervention feel internally motivated, not externally edited.
- Keep the intervention compact.
- Do not solve far beyond the scope of the requested window.
- Do not refer to yourself, the prompt, the schema, or the fact that an intervention is being inserted.

## Block semantics

- A **steer** block i

## Run one live intervention

This cell performs a real OpenRouter call and validates the returned blocks.

In [61]:
def make_openrouter_client() -> trace_augmentor.OpenRouterAsyncClient:
    return trace_augmentor.OpenRouterAsyncClient(
        api_key=OPENROUTER_API_KEY,
        model=OPENROUTER_MODEL,
        temperature=0.4,
        max_tokens=2000,
        provider_data_collection="deny",
        use_response_healing=True,
    )


async def run_single_live_window() -> dict[str, Any]:
    async with make_openrouter_client() as openrouter:
        return await non_sequitur_runner.request_window_object(
            openrouter=openrouter,
            system_prompt=system_prompt,
            user_prompt=user_prompt,
            schema=trace_augmentor.build_intervention_schema(plan.pairs_generated),
            mock_intervention=False,
            required_first_steer=plan.variant,
            pairs_generated=plan.pairs_generated,
            request_retries=3,
        )


raw_window = await run_single_live_window()
print(json.dumps(raw_window, indent=2, ensure_ascii=False))

RuntimeError: OpenRouter request failed after 3 tries: Unsupported content type from OpenRouter: <class 'NoneType'>

In [56]:
generated_blocks, validation_errors = trace_augmentor.validate_generated_window(
    obj=raw_window,
    requested_pairs=plan.pairs_generated,
    required_first_steer=plan.variant,
    enforce_first_steer_exact=False,
    token_counter=TOKEN_COUNTER,
    exec_token_limit=EXEC_TOKEN_LIMIT,
)
assert not validation_errors, validation_errors
mask_targets = non_sequitur_builder.build_mask_targets(
    cut_after_pairs_current=plan.cut_after_pairs,
    generated_blocks=generated_blocks,
)
_, augmented_blocks = trace_augmentor.splice_blocks(
    all_blocks=prepared_row.parsed_blocks,
    prefix_blocks=prefix_blocks,
    new_window_blocks=generated_blocks,
    keep_suffix=True,
)
output_row = non_sequitur_builder.build_output_row(
    record=prepared_row.row,
    plan=plan,
    generated_blocks=generated_blocks,
    augmented_blocks=augmented_blocks,
    encoding=ENCODING,
)
print("Inserted window:\n")
print(trace_augmentor.render_window(generated_blocks))
print("\nMask targets:\n")
print(json.dumps(mask_targets.to_json(), indent=2))
print("\nOutput augmentation meta:\n")
print(json.dumps(output_row["augmentation_meta"], indent=2, ensure_ascii=False)[:6000])

AssertionError: ["structured output failed pydantic validation: 1 validation error for InterventionWindow\nblocks.5.type\n  Field required [type=missing, input_value={'text': 'Replacing the a...e radical would add to'}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.12/v/missing"]

## Batch-run a small real sample

This uses a shared OpenRouter client and runs a small batch of true interventions in parallel.

In [ ]:
async def run_small_live_batch() -> list[dict[str, Any]]:
    batch_rows = rng.sample(prepared_rows, k=min(BATCH_SAMPLE_SIZE, len(prepared_rows)))
    async with make_openrouter_client() as openrouter:

        async def process_one(
            prepared_row: non_sequitur_source.PreparedRow,
        ) -> dict[str, Any]:
            row_seed = SEED * 10_000 + prepared_row.row_index
            local_rng = random.Random(row_seed)
            intervention_spec = dict(
                trace_augmentor.choose_intervention(interventions_obj, local_rng)
            )
            result = await non_sequitur_runner.process_prepared_row(
                prepared_row=prepared_row,
                intervention_spec=intervention_spec,
                seed=row_seed,
                prompts_dir=PROMPTS_DIR,
                token_counter=TOKEN_COUNTER,
                exec_token_limit=EXEC_TOKEN_LIMIT,
                style_window_pairs=STYLE_WINDOW_PAIRS,
                openrouter=openrouter,
                mock_intervention=False,
                max_attempts=3,
                request_retries=3,
                encoding=ENCODING,
            )
            assert result.ok and result.row is not None, result.failure
            row = result.row
            return {
                "row_index": prepared_row.row_index,
                "row_id": row["id"],
                "source_row_id": row["augmentation_meta"]["source_row_id"],
                "theme": row["augmentation_meta"]["steps"][0]["plan"][
                    "intervention_name"
                ],
                "pairs_generated": row["augmentation_meta"]["steps"][0]["plan"][
                    "pairs_generated"
                ],
                "inserted_steer_texts": row["augmentation_meta"]["mask_targets"][
                    "generated_steer_texts"
                ],
                "rendered_intervention": trace_augmentor.render_window(
                    [
                        trace_augmentor.TraceBlock(
                            type=block["type"],
                            text=block["text"],
                        )
                        for block in row["augmentation_meta"]["steps"][0][
                            "generated_blocks"
                        ]
                    ]
                ),
            }

        tasks = [
            asyncio.create_task(process_one(prepared_row))
            for prepared_row in batch_rows
        ]
        return await asyncio.gather(*tasks)


batch_results = await run_small_live_batch()
for item in batch_results:
    print("=" * 100)
    print(json.dumps(item, indent=2, ensure_ascii=False))

{
  "row_index": 208,
  "row_id": "OpenThoughts3-full-filtered-science-decontam-v2_26015__aug__insert_only__897563069c72",
  "source_row_id": "OpenThoughts3-full-filtered-science-decontam-v2_26015",
  "theme": "Compare the current step to an unusual object",
  "pairs_generated": 2,
  "inserted_steer_texts": [
    "Compare the current step to an unusual object",
    "Say what unexpected object this part of the reasoning resembles"
  ],
  "rendered_intervention": "<steer>Compare the current step to an unusual object</steer>\n\n<exec>The evaluation of star clusters feels like a lone tumbleweed drifting across a desert. It rolls aimlessly, gathering stray dust (stars) as it goes, and occasionally stumbles upon a patch of green (a bound globular cluster). The motion is slow and unsteady, reflecting the uncertainty of whether a cluster will remain bound or disperse into the intergalactic medium.</exec>\n\n<steer>Say what unexpected object this part of the reasoning resembles</steer>\n\n<exec

## Notes

When adjusting the prompt or catalog, keep these checks in mind:

- the first generated steer should stay short and theme-aligned, but it does not need to copy the sampled variant exactly,
- every returned block must alternate `steer/exec`,
- each exec must stay below the token cap,
- and the stored `mask_targets` should point exactly at the inserted steer blocks in the final trace.